In [ ]:
import json
from typing import List
from botocore.client import BaseClient
import boto3
from botocore.exceptions import ClientError
import faiss
import numpy as np
import pickle

In [2]:
session = boto3.Session(profile_name="aws_learning")


In [9]:
def create_s3_bucket(bucket_name:str, region:str = "ca-central-1"):
    """
    Create an S3 bucket if it does not already exist.

    Parameters
    ----------
    bucket_name : str
        Preferred S3 bucket name
    region : str
        region
    
    Returns
    -------
    None
    """

    s3 = session.client("s3", region_name=region)
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Bucket '{bucket_name}' already exists.")
        return
    except ClientError:
        pass

    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )

    print(f"Created bucket: {bucket_name}")

In [10]:
#create_s3_bucket(BUCKET_NAME)

Created bucket: researchassistant-rag-prod-embeddings


In [5]:
def get_titan_embedding(
    text: str,
    bedrock_client: BaseClient,
    dimensions: int = 1024,
    normalize: bool = True,
) -> List[float]:
    """
    Generate a vector embedding for input text using Amazon Titan Text Embeddings V2.

    Parameters
    ----------
    text : str
        Text to convert into an embedding vector.
    bedrock_client : botocore.client.BaseClient
        Boto3 client configured for the Amazon Bedrock Runtime service.
    dimensions : int, optional
        Dimensionality of the output embedding vector. Defaults to 1024.
    normalize : bool, optional
        Whether to normalize the output embedding vector. Defaults to True.

    Returns
    -------
    list of float
        Embedding vector representing the input text.
    """

    model_id = "amazon.titan-embed-text-v2:0"

    body = json.dumps({
        "inputText": text,
        "dimensions": dimensions,
        "normalize": normalize,
    })

    response = bedrock_client.invoke_model(
        modelId=model_id,
        body=body,
        accept="application/json",
        contentType="application/json",
    )

    try:
        response_body = json.loads(response["body"].read())
    except Exception as e:
        raise RuntimeError(f"Failed to parse Bedrock response: {response}") from e

    embedding = response_body.get("embedding")
    if embedding is None:
        raise RuntimeError(f"Unexpected Bedrock response: {response_body}")

    return embedding

In [4]:
bedrock_client = session.client(
    service_name="bedrock-runtime",
    region_name="ca-central-1",
)

[-0.030301740393042564,
 0.05391927435994148,
 -0.006322146393358707,
 -0.05570172891020775,
 0.06238593906164169,
 -0.07040698826313019,
 0.05280524119734764,
 0.01715613342821598,
 -0.04924032837152481,
 -0.1657683551311493,
 0.03364384546875954,
 0.03564910590648651,
 0.09625259041786194,
 0.031638581305742264,
 -0.014928063377737999,
 -0.06817891448736191,
 0.013257011771202087,
 0.015819290652871132,
 -0.012477187439799309,
 0.11274030059576035,
 0.016376309096813202,
 0.01425964292138815,
 0.12655432522296906,
 -0.06550523638725281,
 -0.038768403232097626,
 -0.11853328347206116,
 -0.03453507274389267,
 -0.06060348078608513,
 -0.047457873821258545,
 -0.09848065674304962,
 -0.016153501346707344,
 0.13814029097557068,
 0.033866651356220245,
 0.09714381396770477,
 0.04589822515845299,
 0.007742540445178747,
 0.04946313798427582,
 0.02038683369755745,
 -0.061049096286296844,
 0.010304819792509079,
 0.09045960754156113,
 -0.04545261338353157,
 -0.004846050404012203,
 -0.082884177565574

In [6]:
embedding = get_titan_embedding(
    text="AWS Titan embedding example",
    bedrock_client=bedrock_client,
    dimensions=256,
)
embedding

[-0.030301740393042564,
 0.05391927435994148,
 -0.006322146393358707,
 -0.05570172891020775,
 0.06238593906164169,
 -0.07040698826313019,
 0.05280524119734764,
 0.01715613342821598,
 -0.04924032837152481,
 -0.1657683551311493,
 0.03364384546875954,
 0.03564910590648651,
 0.09625259041786194,
 0.031638581305742264,
 -0.014928063377737999,
 -0.06817891448736191,
 0.013257011771202087,
 0.015819290652871132,
 -0.012477187439799309,
 0.11274030059576035,
 0.016376309096813202,
 0.01425964292138815,
 0.12655432522296906,
 -0.06550523638725281,
 -0.038768403232097626,
 -0.11853328347206116,
 -0.03453507274389267,
 -0.06060348078608513,
 -0.047457873821258545,
 -0.09848065674304962,
 -0.016153501346707344,
 0.13814029097557068,
 0.033866651356220245,
 0.09714381396770477,
 0.04589822515845299,
 0.007742540445178747,
 0.04946313798427582,
 0.02038683369755745,
 -0.061049096286296844,
 0.010304819792509079,
 0.09045960754156113,
 -0.04545261338353157,
 -0.004846050404012203,
 -0.082884177565574

In [22]:


def build_faiss_index(embeddings: np.ndarray):
    """
    Builds a cosine-similarity FAISS index.
    """
    dim = len(embeddings[0])

    # Normalize vectors → cosine similarity
    #faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    return index


def save_index(index, metadata: list[dict], index_path: str, meta_path: str):
    """
    Saves FAISS index + metadata locally.
    """
    faiss.write_index(index, index_path)

    with open(meta_path, "wb") as f:
        pickle.dump(metadata, f)


s3 = session.client("s3", region_name="ca-central-1")
def upload_file(local_path: str, bucket: str, s3_key: str):
    s3.upload_file(local_path, bucket, s3_key)
    print(f"Uploaded {s3_key}")

In [13]:
with open("data_to_embed.json", "r") as f:
    data = json.load(f)

data

[{'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
  'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
  'sections': ['Abstract - Results'],
  'strength': 'strong',
  'source_chunks': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe_chunk_000'],
  'study_quality': {'study_design': 'rct',
   'sample_size': 'small',
   'bias_risk': 'moderate',
   'evidence_strength': 'strong',
   'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported

In [14]:
claims_list = [
    {
        "id": f"{claim['paper_id']}::{idx:03d}",
        "embedding_text": f"Claim: {claim['claim']} Evidence: {claim['evidence']}",
        "metadata": {
            "paper_id": claim['paper_id'],
            "claim": claim['claim'],
            "evidence": claim['evidence'],
            "section": claim['sections'],
            "study_quality": claim['study_quality'],
            "title": claim['title']
        }
    }
    for idx, claim in enumerate(data, start=1)
]

claims_list

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::001',
  'embedding_text': "Claim: The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group Evidence: ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
   'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
   'section': ['Abstract - Results'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderat

In [ ]:
BUCKET = "researchassistant-rag-prod-embeddings"
INDEX_PATH = "claims_test.index"
META_PATH = "meta_test.pkl"

In [16]:
texts = [
    f"{c['embedding_text']}" for c in claims_list
]
texts

["Claim: The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group Evidence: ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.']",
 "Claim: The reduction of diastolic blood pressure was 5.351 ± 5.643 mm Hg in the DASH group and 9.459 ± 4.375 mm Hg in the DASH + TRE group Evidence: ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.']",
 "Claim: Subjects in DASH + TRE group had decreased extracellular water and increased urinary Na+ excretion Evidence: ['Subjects in DASH + TRE group had decreased extracellular water and increased urinary Na+ excretion.']",
 "Claim: The DASH + TRE group showed significantly greater reductio

In [18]:
embeddings = [
    get_titan_embedding(text=x,
    bedrock_client=bedrock_client,
    dimensions=256,) for x in texts
]
embeddings

[[-0.08323096483945847,
  -0.032864563167095184,
  -0.005955486558377743,
  -0.013612541370093822,
  0.019252022728323936,
  -0.1073446124792099,
  0.12601323425769806,
  -0.09217634797096252,
  -0.024113643914461136,
  -0.007195200305432081,
  -0.006417340599000454,
  -0.06028411164879799,
  0.06495127081871033,
  0.05289444699883461,
  0.002406502841040492,
  0.05561695247888565,
  0.012737449258565903,
  0.028975265100598335,
  -0.08673132956027985,
  0.002248500008136034,
  -0.009820476174354553,
  0.008459221571683884,
  0.07700809091329575,
  0.05833946168422699,
  -0.0976213663816452,
  0.001993265002965927,
  0.007875827141106129,
  -0.08439775556325912,
  0.06806270778179169,
  -0.02226622775197029,
  0.05133872479200363,
  0.05094979703426361,
  -0.04531031474471092,
  -0.027030616998672485,
  0.04317120090126991,
  0.03636493161320686,
  0.0820641741156578,
  0.05172765627503395,
  0.06534019857645035,
  0.002357886638492346,
  0.030725449323654175,
  -0.04005976393818855,
 

In [21]:
len(embeddings[0])

256

In [28]:
mbx = np.array(embeddings, dtype="float32")
mbx.shape

(9, 256)

In [29]:
index = build_faiss_index(mbx)
index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x10e6a0630> >

In [30]:
save_index(index, claims_list, INDEX_PATH, META_PATH)

In [33]:
upload_file(INDEX_PATH, BUCKET, "faiss/claims.index")
upload_file(META_PATH, BUCKET, "faiss/claims_meta.pkl")

Uploaded faiss/claims.index
Uploaded faiss/claims_meta.pkl


In [46]:
downloaded_index_path = "downloads/faiss_claims.index"
downloaded_metadata_path = "downloads/metadata.pkl"

s3.download_file(BUCKET, Key = "faiss/claims.index", Filename= downloaded_index_path )
s3.download_file(BUCKET, "faiss/claims_meta.pkl", downloaded_metadata_path)

In [47]:
downloaded_index = faiss.read_index(downloaded_index_path)
with open(downloaded_metadata_path, "rb") as f:
    downloaded_metadata = pickle.load(f)

In [56]:
user_query = "fasting"

user_embeddings = get_titan_embedding(
    text=user_query,
    bedrock_client=bedrock_client,
    dimensions=256)

In [58]:
user_embeddings = np.array(user_embeddings, dtype="float32").reshape(1, -1)
user_embeddings.shape

(1, 256)

In [59]:
scores, indices = downloaded_index.search(user_embeddings, 3)

In [62]:
scores[0]

array([0.15112162, 0.14141172, 0.12043948], dtype=float32)

In [66]:
indices[0]

array([6, 7, 1])

In [65]:
results = []
for idx, score in zip(indices[0], scores[0]):
    item = downloaded_metadata[idx].copy()
    item["similarity_score"] = float(score)
    results.append(item)
results

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

In [67]:
len(downloaded_metadata)

9

In [69]:
downloaded_index.ntotal

9